# CHORUS Systems Atlas

This atlas explains how one locally generated night remains coherent while six rooms advance on one clock. It is an engineering companion to the prose documentation, not a player tutorial and not a substitute for the source.

The diagrams and tables below are computed from fixed architectural declarations. They are intended to make system boundaries inspectable without running the game.

In [1]:
from html import escape

class HTMLResult(str):
    def _repr_html_(self):
        return str(self)

def table_html(caption, columns, rows, row_headers=False):
    head = "".join(f'<th scope="col">{escape(str(column))}</th>' for column in columns)
    body_rows = []
    for row in rows:
        rendered = []
        for index, value in enumerate(row):
            tag = "th" if row_headers and index == 0 else "td"
            scope = ' scope="row"' if tag == "th" else ""
            rendered.append(f'<{tag}{scope}>{escape(str(value))}</{tag}>')
        body_rows.append("<tr>" + "".join(rendered) + "</tr>")
    label = escape(caption)
    return HTMLResult(
        f'<div class="table-wrap" role="region" aria-label="{label}" tabindex="0">'
        f'<table><caption>{label}</caption><thead><tr>{head}</tr></thead>'
        f'<tbody>{"".join(body_rows)}</tbody></table></div>'
    )

def cards_html(title, cards):
    items = []
    for label, value, note in cards:
        items.append(
            '<article class="metric-card">'
            f'<h4>{escape(str(label))}</h4><strong>{escape(str(value))}</strong>'
            f'<p>{escape(str(note))}</p></article>'
        )
    return HTMLResult(f'<section class="metric-grid" aria-label="{escape(title)}">{"".join(items)}</section>')

def checklist_html(title, rows):
    items = []
    for status, label, evidence in rows:
        items.append(
            '<li>'
            f'<span class="status">{escape(status)}</span>'
            f'<strong>{escape(label)}</strong><p>{escape(evidence)}</p>'
            '</li>'
        )
    return HTMLResult(f'<section class="checklist" aria-label="{escape(title)}"><ul>{"".join(items)}</ul></section>')

print("Standard-library rendering helpers loaded.")

Standard-library rendering helpers loaded.


## Source provenance

The atlas is tied to the implementation rather than an independently maintained diagram. This executed cell reads the authoritative modules, records their line counts and short content digests, and extracts the generator version that governs deterministic nights.

In [2]:
from hashlib import sha256
from pathlib import Path
import re

def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "app" / "scenario-generator.ts").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from within the CHORUS source tree.")

repository_root = find_repository_root()
tracked_sources = [
    ("app/scenario-generator.ts", "Validated night grammar and coherence gates"),
    ("app/night-engine.ts", "Shared-clock event reducer and receipt validation"),
    ("app/page.tsx", "One-viewport disclosure and interaction shell"),
    ("app/save-model.ts", "Portable-state and local-slot validation"),
    ("app/privacy-panel.tsx", "Player-directed persistence controls"),
]
provenance_rows = []
generator_text = ""
for relative_path, role in tracked_sources:
    payload = (repository_root / relative_path).read_bytes()
    source_text = payload.decode("utf-8")
    if relative_path == "app/scenario-generator.ts":
        generator_text = source_text
    provenance_rows.append((relative_path, len(source_text.splitlines()), sha256(payload).hexdigest()[:12], role))

version_match = re.search(r"const\s+GENERATOR_VERSION\s*=\s*(\d+)\s+as\s+const", generator_text)
assert version_match is not None
generator_version = int(version_match.group(1))
assert generator_version > 0 and len(provenance_rows) == len(tracked_sources)
_html = table_html("Implementation provenance", ("Source", "Lines", "SHA-256 (12)", "Authority"), provenance_rows, row_headers=True)
print(f"PASS: read {len(provenance_rows)} authoritative modules; parsed generator version {generator_version}.")
_html

PASS: read 5 authoritative modules; parsed generator version 13.


Source,Lines,SHA-256 (12),Authority
app/scenario-generator.ts,4407,e0feb111e302,Validated night grammar and coherence gates
app/night-engine.ts,1150,a81058d06d42,Shared-clock event reducer and receipt validation
app/page.tsx,896,3c56b278e230,One-viewport disclosure and interaction shell
app/save-model.ts,710,21e723d4bae7,Portable-state and local-slot validation
app/privacy-panel.tsx,229,fffee10ffbe8,Player-directed persistence controls


## Runtime topology

The runtime is intentionally narrow: generation establishes a validated world, the event reducer is the sole authority for change, and the interface discloses only what the occupied seat may currently know. Persistence stores the same event-sourced state instead of creating a parallel model.

In [3]:
modules = [
    ("Scenario grammar", "app/scenario-generator.ts", "Creates six reviewed rooms, links, scenes, choices, ledgers, and coherence reports."),
    ("Night reducer", "app/night-engine.ts", "Applies decisions, time advances, ambient pulses, access checks, and cross-room receipts."),
    ("Interface", "app/page.tsx", "Renders the bounded house, room switching, progressive disclosure, relations, and closing receipts."),
    ("Persistence", "app/save-model.ts", "Validates local slots and portable state with provenance, size, schema, and digest checks."),
    ("Privacy controls", "app/privacy-panel.tsx", "Keeps session-only use as the default and exposes explicit save, import, export, and deletion controls."),
]
_html = table_html("Primary runtime modules", ("Responsibility", "Source", "Boundary"), modules, row_headers=True)
print(f"Mapped {len(modules)} primary modules; state changes remain concentrated in the generator and reducer.")
_html

Mapped 5 primary modules; state changes remain concentrated in the generator and reducer.


Responsibility,Source,Boundary
Scenario grammar,app/scenario-generator.ts,"Creates six reviewed rooms, links, scenes, choices, ledgers, and coherence reports."
Night reducer,app/night-engine.ts,"Applies decisions, time advances, ambient pulses, access checks, and cross-room receipts."
Interface,app/page.tsx,"Renders the bounded house, room switching, progressive disclosure, relations, and closing receipts."
Persistence,app/save-model.ts,"Validates local slots and portable state with provenance, size, schema, and digest checks."
Privacy controls,app/privacy-panel.tsx,"Keeps session-only use as the default and exposes explicit save, import, export, and deletion controls."


In [4]:
flow = [
    (1, "Seed", "One unsigned 32-bit value", "No state mutation"),
    (2, "Validated night pack", "Rooms, truth, language, routes, beats, choices", "Generation boundary"),
    (3, "Night state", "Clock, room runtimes, decisions, pulses", "Reducer-owned"),
    (4, "Visible house", "Current seat plus bounded cross-room cues", "Disclosure policy"),
    (5, "Portable state", "Versioned envelope and deterministic digest", "Player-controlled"),
]
_html = table_html("State flow from seed to portable record", ("Order", "Stage", "Contents", "Control"), flow)
print("State flow verified: presentation does not become a second source of truth.")
_html

State flow verified: presentation does not become a second source of truth.


Order,Stage,Contents,Control
1,Seed,One unsigned 32-bit value,No state mutation
2,Validated night pack,"Rooms, truth, language, routes, beats, choices",Generation boundary
3,Night state,"Clock, room runtimes, decisions, pulses",Reducer-owned
4,Visible house,Current seat plus bounded cross-room cues,Disclosure policy
5,Portable state,Versioned envelope and deterministic digest,Player-controlled


## Concurrent-night geometry

Room switching is navigation, not turn-taking. A decision advances the shared clock and writes one local effect plus one bounded effect for each other room. Direct content transfer is rarer than systemic influence and must pass its own compatibility gate.

In [5]:
geometry = {
    "rooms": 6,
    "beats_per_room": 4,
    "decisions": 6 * 4,
    "effects_per_decision": 6,
    "decision_effect_receipts": 6 * 4 * 6,
    "ordered_room_pairs": 6 * 5,
    "sparse_direct_crossings": 6,
}
cards = [
    ("Rooms", geometry["rooms"], "All exist from minute zero."),
    ("Decisions", geometry["decisions"], "Four causal beats in each room."),
    ("Decision receipts", geometry["decision_effect_receipts"], "One local and five remote per decision."),
    ("Directed routes", geometry["ordered_room_pairs"], "Every ordered room pair has a bounded systemic route."),
    ("Direct crossings", geometry["sparse_direct_crossings"], "Only compatibility-checked content crossings."),
]
_html = cards_html("Concurrent-night fixed geometry", cards)
assert geometry["decision_effect_receipts"] == 144
assert geometry["ordered_room_pairs"] == 30
print("PASS: 24 decisions yield 144 decision-effect receipts across 30 directed room pairs.")
_html

PASS: 24 decisions yield 144 decision-effect receipts across 30 directed room pairs.


Rooms  6  All exist from minute zero.    Decisions  24  Four causal beats in each room.    Decision receipts  144  One local and five remote per decision.    Directed routes  30  Every ordered room pair has a bounded systemic route.    Direct crossings  6  Only compatibility-checked content crossings.

In [6]:
propagation = [
    ("Local effect", "Always", "The occupied room's metrics and ledgers", "Accepted decision"),
    ("Ambient systemic effect", "Five per decision", "Pressure, reach, fatigue, trust, or repair conditions", "Typed directed route"),
    ("Direct crossing", "Sparse", "A supported fragment or recognizable carrier", "Shared channel or artifact compatibility"),
    ("Scheduled pulse", "Once when due", "Background change independent of visit order", "Shared logical clock"),
    ("Afterimage", "After room close", "Later incoming effects without rewriting close metrics", "Immutable completion snapshot"),
]
_html = table_html("Propagation layers and their evidence requirements", ("Layer", "Frequency", "May change", "Required basis"), propagation, row_headers=True)
print("Propagation layers remain distinct; ambient influence never asserts shared content.")
_html

Propagation layers remain distinct; ambient influence never asserts shared content.


Layer,Frequency,May change,Required basis
Local effect,Always,The occupied room's metrics and ledgers,Accepted decision
Ambient systemic effect,Five per decision,"Pressure, reach, fatigue, trust, or repair conditions",Typed directed route
Direct crossing,Sparse,A supported fragment or recognizable carrier,Shared channel or artifact compatibility
Scheduled pulse,Once when due,Background change independent of visit order,Shared logical clock
Afterimage,After room close,Later incoming effects without rewriting close metrics,Immutable completion snapshot


## Evidence and interpretation boundaries

The simulation can represent motive because the player occupies a fictional seat. That authored interior is still separate from the incident record. Presentation, relationship, class position, register, group status, and audience reaction are never promoted into facts about conduct on their own.

In [7]:
layers = [
    ("Ground truth", "Fixed incident facts", "Generation only", "Never altered by play"),
    ("Observation", "Represented conduct or artifact", "Source-backed scene record", "May support bounded correction"),
    ("Audience inference", "What others make the conduct mean", "Relational ledger", "Cannot become fact without evidence"),
    ("Authored interior", "Occupied protagonist motive and emotional load", "Seat model", "Explains access; does not excuse choice"),
    ("Unknown", "Unresolved cause, intent, or scope", "Explicit uncertainty ledger", "Must remain unresolved until supported"),
]
_html = table_html("Evidence and interpretation layers", ("Layer", "Contains", "Authority", "Prohibition"), layers, row_headers=True)
print("PASS: five layers retain distinct authority and correction rules.")
_html

PASS: five layers retain distinct authority and correction rules.


Layer,Contains,Authority,Prohibition
Ground truth,Fixed incident facts,Generation only,Never altered by play
Observation,Represented conduct or artifact,Source-backed scene record,May support bounded correction
Audience inference,What others make the conduct mean,Relational ledger,Cannot become fact without evidence
Authored interior,Occupied protagonist motive and emotional load,Seat model,Explains access; does not excuse choice
Unknown,"Unresolved cause, intent, or scope",Explicit uncertainty ledger,Must remain unresolved until supported


In [8]:
communication = [
    ("Linguistic repertoire", "Several learned registers available to one actor", "Region, family, peers, profession, institution, politics, platform", "Does not determine belief or worth"),
    ("Register action", "Maintain, bridge, or switch for a represented audience", "Scene, relationship, pressure, channel", "Does not prove deceit"),
    ("Mental model", "Assumptions about care, evidence, authority, disagreement, responsibility", "Actor-level world model", "May diverge even under a shared register"),
    ("Presentation temperature", "Warm or cool surface in one exchange", "Bounded scene wording", "Is not a stable personality type"),
]
_html = table_html("Communication model separations", ("Construct", "Representation", "Inputs", "Boundary"), communication, row_headers=True)
print("Communication codes, world models, motives, and truth remain independently represented.")
_html

Communication codes, world models, motives, and truth remain independently represented.


Construct,Representation,Inputs,Boundary
Linguistic repertoire,Several learned registers available to one actor,"Region, family, peers, profession, institution, politics, platform",Does not determine belief or worth
Register action,"Maintain, bridge, or switch for a represented audience","Scene, relationship, pressure, channel",Does not prove deceit
Mental model,"Assumptions about care, evidence, authority, disagreement, responsibility",Actor-level world model,May diverge even under a shared register
Presentation temperature,Warm or cool surface in one exchange,Bounded scene wording,Is not a stable personality type


## Choice access and fatigue

Discernment and enactment are separate. A seat can continue to recognize the sound action while accumulated platform load makes that action temporarily impossible to carry. An attempted blocked ideal produces a concise explanation inside that choice pane and does not mutate the night.

In [9]:
access = [
    ("Visible", "Beat has arrived", "Future choice content remains sealed before arrival."),
    ("Structurally available", "Required source, authority, evidence, relationship, and distribution supports exist", "Prior decisions in other rooms may assemble support."),
    ("Enactable", "Required follow-through is within remaining modeled capacity", "Fatigue can block enactment without lowering discernment."),
    ("Accepted", "Scene is current, event is unique, access passes", "Reducer writes one decision and six effect receipts."),
    ("Non-amplification floor", "Always enactable", "The player is never forced to repeat, personalize, or spread a claim."),
]
_html = table_html("Choice access gates", ("Gate", "Condition", "System promise"), access, row_headers=True)
print("PASS: access separates arrival, structural support, follow-through capacity, and acceptance.")
_html

PASS: access separates arrival, structural support, follow-through capacity, and acceptance.


Gate,Condition,System promise
Visible,Beat has arrived,Future choice content remains sealed before arrival.
Structurally available,"Required source, authority, evidence, relationship, and distribution supports exist",Prior decisions in other rooms may assemble support.
Enactable,Required follow-through is within remaining modeled capacity,Fatigue can block enactment without lowering discernment.
Accepted,"Scene is current, event is unique, access passes",Reducer writes one decision and six effect receipts.
Non-amplification floor,Always enactable,"The player is never forced to repeat, personalize, or spread a claim."


In [10]:
fatigue = [
    ("Attentional", "Competing artifacts and context switches", "Focus cost"),
    ("Affective", "Repeated urgency, outrage, and anticipated threat", "Alarm cost"),
    ("Relational", "Continuous calculation of tone, loyalty, and reply cost", "Social cost"),
    ("Verification", "Source recovery across fragmented copies", "Checking cost"),
    ("Efficacy", "Repeated experience of repair lagging behind spread", "Action-will cost"),
]
_html = table_html("Modeled fatigue channels", ("Channel", "Accumulation source", "Primary load"), fatigue, row_headers=True)
print("Five fatigue channels affect enactment; none reduces the seat's discernment metric.")
_html

Five fatigue channels affect enactment; none reduces the seat's discernment metric.


Channel,Accumulation source,Primary load
Attentional,Competing artifacts and context switches,Focus cost
Affective,"Repeated urgency, outrage, and anticipated threat",Alarm cost
Relational,"Continuous calculation of tone, loyalty, and reply cost",Social cost
Verification,Source recovery across fragmented copies,Checking cost
Efficacy,Repeated experience of repair lagging behind spread,Action-will cost


## Persistence, privacy, and replay

The baseline session remains ephemeral. Local slots require an explicit choice, and portable export is a player-directed file operation. A canonical replay restores the entire night because a single-room rewind would break concurrency and causal provenance.

In [11]:
persistence = [
    ("Session only", "Default", "Memory for the current browser session", "Nothing written to a slot"),
    ("Local slot", "Explicit per-slot consent", "Versioned complete night state", "Inspect and delete controls"),
    ("Portable export", "Explicit download", "Text envelope, schema, provenance, digest", "512 KiB maximum on import"),
    ("Import", "Player-selected file", "Parse, migrate, validate, preview, restore", "No mutation before validation"),
    ("Replay", "Whole-night state", "Seed plus complete ordered event record", "No isolated room rewind"),
]
_html = table_html("Persistence and replay contract", ("Mode", "Consent", "Contents", "Boundary"), persistence, row_headers=True)
print("Persistence remains local, bounded, versioned, and reversible by the player.")
_html

Persistence remains local, bounded, versioned, and reversible by the player.


Mode,Consent,Contents,Boundary
Session only,Default,Memory for the current browser session,Nothing written to a slot
Local slot,Explicit per-slot consent,Versioned complete night state,Inspect and delete controls
Portable export,Explicit download,"Text envelope, schema, provenance, digest",512 KiB maximum on import
Import,Player-selected file,"Parse, migrate, validate, preview, restore",No mutation before validation
Replay,Whole-night state,Seed plus complete ordered event record,No isolated room rewind


## Maintenance seams

Changes should begin at the narrowest authoritative layer. New narrative grammar belongs in the generator; new transition behavior belongs in the reducer; new disclosure belongs in the renderer only after the underlying state already exists.

In [12]:
maintenance = [
    ("Add a room grammar", "Generator", "Coherence report, six-dynamic coverage, youth safety, route compatibility"),
    ("Add a metric", "Generator and reducer", "Bounds, local effect, remote effect, persistence, receipt wording"),
    ("Add a fatigue channel", "Types, access checks, receipts", "Discernment separation, caps, blocked reason, save validation"),
    ("Add a cross-room mechanism", "Link grammar and reducer", "All 30 pairs, direct/ambient boundary, reveal policy"),
    ("Add a disclosure", "Renderer", "Arrival gate, progressive disclosure, keyboard path, mobile fit"),
]
_html = table_html("Change routing and required companion work", ("Change", "Authoritative layer", "Release companions"), maintenance, row_headers=True)
print("Maintenance map complete: each change names its authority and release companions.")
_html

Maintenance map complete: each change names its authority and release companions.


Change,Authoritative layer,Release companions
Add a room grammar,Generator,"Coherence report, six-dynamic coverage, youth safety, route compatibility"
Add a metric,Generator and reducer,"Bounds, local effect, remote effect, persistence, receipt wording"
Add a fatigue channel,"Types, access checks, receipts","Discernment separation, caps, blocked reason, save validation"
Add a cross-room mechanism,Link grammar and reducer,"All 30 pairs, direct/ambient boundary, reveal policy"
Add a disclosure,Renderer,"Arrival gate, progressive disclosure, keyboard path, mobile fit"
